# Financial News Sentiment Prediction -- End-to-End Pipeline
Classify finance-related tweets as Bearish, Bullish or Neutral.
Dataset: zroshot/twitter-financial-news-sentiment -- 9938 train / 2486 val.
Labels: 0=Bearish, 1=Bullish, 2=Neutral

## 1. Setup & Imports

In [ ]:
import os, random, pickle
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from src.data_loader import load_hf_dataset, clean_tweet, Vocabulary, make_dataloaders, describe_split, LABEL_NAMES
from src.models import build_model
from src.train import train_model, compute_class_weights, predict_text
from src.evaluation import plot_confusion_matrix, plot_training_curves, classification_table, comparison_bar, summary_table
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MODELS_DIR = Path('models'); MODELS_DIR.mkdir(exist_ok=True)
REPORTS_DIR = Path('reports'); REPORTS_DIR.mkdir(exist_ok=True)
print('Device:', DEVICE)

## 2. Load Dataset

In [ ]:
train_df, val_df = load_hf_dataset()
train_dist = describe_split(train_df, 'TRAIN')
val_dist = describe_split(val_df, 'VAL')
with open(MODELS_DIR/'val_distribution.pkl','wb') as f: pickle.dump(val_dist,f)

## 3. Preprocess

In [ ]:
MAX_LEN = 48; BATCH_SIZE = 64
vocab = Vocabulary.build(train_df['text'].tolist(), min_freq=2, max_size=30000)
with open(MODELS_DIR/'vocab.pkl','wb') as f: pickle.dump(vocab,f)
train_loader, val_loader = make_dataloaders(train_df, val_df, vocab, BATCH_SIZE, MAX_LEN)
class_weights = compute_class_weights(train_df['label'].tolist())
print('Vocab:', len(vocab), 'weights:', class_weights.tolist())

## 4. Train RNN Baselines

In [ ]:
results = {}; histories = {}; trained = {}
for model_name in ['rnn', 'lstm', 'gru']:
    print(f'\n=== Training {model_name.upper()} ===')
    torch.manual_seed(SEED)
    model = build_model(model_name, vocab_size=len(vocab))
    model, history, stats = train_model(model, train_loader, val_loader, epochs=8, lr=1e-3, class_weights=class_weights, patience=3, device=DEVICE)
    results[model_name] = stats; histories[model_name] = history; trained[model_name] = model
    torch.save(model.state_dict(), MODELS_DIR / f'{model_name}_best.pt')

## 5. Evaluate

In [ ]:
for name, hist in histories.items():
    plot_training_curves(hist, title_prefix=name.upper(), savepath=REPORTS_DIR/f'curves_{name}.png')
for name, r in results.items():
    plot_confusion_matrix(r['confusion_matrix'], title=f'{name.upper()} F1={r['val_f1']:.3f}', savepath=REPORTS_DIR/f'cm_{name}.png')
    print(classification_table(r['y_true'], r['y_pred']))
plt.show()
final_summary = summary_table(results)
print(final_summary)
with open(MODELS_DIR/'results_summary.pkl','wb') as f: pickle.dump(final_summary,f)
comparison_bar(results, metric='val_f1', savepath=REPORTS_DIR/'compare_f1.png')
plt.show()

## 6. (Optional) Fine-tune BERT -- uncomment to run

In [ ]:
# from src.bert_finetune import finetune_bert
# bert_model, bert_tok, bert_stats = finetune_bert(train_df, val_df, model_name='distilbert-base-uncased', output_dir=str(MODELS_DIR/'bert'), epochs=3)
# bert_model.save_pretrained(MODELS_DIR/'bert_final')
# bert_tok.save_pretrained(MODELS_DIR/'bert_final')
# results['bert'] = bert_stats
# print(classification_table(bert_stats['y_true'], bert_stats['y_pred']))

## 7. Error Analysis & Inference Test

In [ ]:
best_name = max({'nn','lstm','gru'}, key=lambda k: results[k]['val_f1'])
test_s = ['Apple beats earnings expectations, stock surges', 'Tesla misses Q3 delivery, shares tumble 8%', 'Fed holds rates steady at 5.25%']
for s in test_s:
    lid, probs = predict_text(s, trained[best_name], vocab, max_len=MAX_LEN, device=DEVICE)
    print(f'{LABEL_NAMES[lid]:<8} ({probs[lid]:.2f}) | {s}')